In [21]:
import sys
from hra_amap.registration.organ import Organ
from hra_amap.registration.tissue import TissueBlock
from hra_amap.registration.pipeline import Pipeline
from hra_amap.registration.dataclass import Projection
from hra_amap.utils.conversions import to_pointcloud

import hra_api_client
from hra_api_client.api import v1_api
from hra_amap.utils.io import read_yaml
from scripts.constants import ConfigKeys

import time
import trimesh
import numpy as np
from copy import deepcopy
from tqdm.auto import tqdm
from pathlib import Path
 

In [22]:
config = Path("../input-data/millitome/pancreas-female-vu/v1.0/config.yaml")
config_dict = read_yaml(config)
config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.SOURCE] = config.parent / config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.SOURCE]
config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.TARGET] = config.parent / config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.TARGET]

##### Set up the CCF API

In [4]:
configuration = hra_api_client.Configuration(
    host = "https://apps.humanatlas.io/api" 
)

api_client = hra_api_client.ApiClient(configuration)
api_instance = v1_api.V1Api(api_client)

##### Connect to Database

In [5]:
db_ready = False
result = None
while not db_ready:
    result = api_instance.db_status()
    if result.status == 'Ready':
        db_ready = True
    else:
        print('Database not ready yet! Retrying...', result)
        time.sleep(2)
print('Database ready!\n', result)

Database ready!
 status='Ready' checkback=3600000 load_time=22594 message='Database successfully loaded'


##### Download Registered Tissue Block Data

In [7]:
try:
    api_response = api_instance.hubmap_rui_locations(cache=True)
except hra_api_client.ApiException as e:
    print("Exception when calling DefaultApi->aggregate_results: %s\n" % e)

##### Filter Tissue Blocks

In [35]:
def filter(data, filter):
  filtered = []
  for donor in tqdm(data):
    filtered_samples = []
    for sample in donor['samples']:
      # get the organ the current sample is registered to
      target = sample['rui_location']['placement']['target'].split('#')[-1]
      # filter
      filter_str = ''.join([filter['gender'], filter['laterality'], filter['name']])
      if filter['version'] == 'All':
        if filter_str in target:
          filtered_samples.append(sample)
      if filter['version'] == 'Latest':
        if filter_str == target:
          filtered_samples.append(sample)
      else:
        filter_str = filter_str + filter['version']
        if filter_str == target:
          filtered_samples.append(sample)
    # add to filtered if valid samples are found
    if filtered_samples:
      donor['samples'] = filtered_samples
      filtered.append(donor)
  
  return filtered

In [36]:
# organ = {'name': 'Pancreas', 
#          'sex': 'Female',
#          'version': 'V1.0'}

In [37]:
# TODO : need to check purpose of filtering and decide on parameter to be considered for filtering(exisiting paramter are not present in the api reposnse)

# result = filter(deepcopy(api_response['@graph']), organ)
# print(result)

  0%|          | 0/126 [00:00<?, ?it/s]

{'@id': 'https://doi.org/10.1016/j.cell.2022.12.028/CRC1#Donor', 'samples': [{'@id': 'https://doi.org/10.1016/j.cell.2022.12.028/CRC1#Donor_TissueBlock1', '@type': 'Sample', 'donor': 'https://doi.org/10.1016/j.cell.2022.12.028/CRC1#Donor', 'datasets': [], 'rui_location': {'@id': 'http://purl.org/ccf/1.5/615aa60c-0a0a-4614-bae2-f85112479919', 'placement': {'@id': 'http://purl.org/ccf/1.5/615aa60c-0a0a-4614-bae2-f85112479919_placement', '@type': 'SpatialPlacement', 'source': 'http://purl.org/ccf/1.5/615aa60c-0a0a-4614-bae2-f85112479919', 'target': 'http://purl.org/ccf/latest/ccf.owl#VHMColon', 'rotation_order': 'XYZ', 'rotation_units': 'degree', 'scaling_units': 'ratio', 'translation_units': 'millimeter', 'x_rotation': 68, 'x_scaling': 1, 'x_translation': 49.126, 'y_rotation': -82, 'y_scaling': 1, 'y_translation': 239.65, 'z_rotation': 1, 'z_scaling': 1, 'z_translation': 140.422, 'creation_date': '2023-08-04'}, '@type': 'SpatialEntity', 'ccf_annotations': ['http://purl.obolibrary.org/obo

KeyError: 'provider_name'

##### Create `TissueBlock`(s)

In [38]:
tissue_blocks = []
target_name = 'VHFRightKidneyV1.1'

for donor in result:
    blocks = [TissueBlock.from_sample(sample, donor, target_name) for sample in donor['samples']]
    tissue_blocks.extend(blocks)

In [39]:
print(f'Found {len(tissue_blocks)} Tissue Blocks')

Found 0 Tissue Blocks


##### Project

In [40]:
# load saved projections (replace this with the saved projections path)
projections = Projection.load('../raw-data/millitome/pancreas-female-vu/v1.0/projections.pickle')

In [41]:
# project
projected_blocks = [projections.project(block) for block in deepcopy(tissue_blocks)]

# create an oriented bounding box around the projected blocks
projected_blocks = [block.bounding_box_oriented for block in projected_blocks]

##### Visualization

In [42]:
# colors
mustard = np.array([225, 173, 1, 255], dtype=np.uint8)
reddishpink = np.array([222, 49, 99, 100], dtype=np.uint8)

In [43]:
# load source organ
source =  Organ(path=config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.SOURCE], target_name = config_dict[ConfigKeys.TARGET_NAME], metadata = config_dict)

# concatenate original tissue blocks before displaying (for efficiency reasons)
tissue_blocks_concatenated = trimesh.util.concatenate(tissue_blocks)

# add color
source.visual.vertex_colors = reddishpink
tissue_blocks_concatenated.visual.vertex_colors = mustard

# create scene with the blocks on the source organ
source_with_tbs = trimesh.Scene([source, tissue_blocks_concatenated])

# show
source_with_tbs.show()

In [44]:
# load the target organ
target = target = Organ(path=config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.TARGET], target_name = config_dict[ConfigKeys.TARGET_NAME], metadata = config_dict)

# concatenate original tissue blocks before displaying (for efficiency reasons)
projected_blocks_concatenated = deepcopy(trimesh.util.concatenate(projected_blocks))

# add color
target.visual.vertex_colors = reddishpink
projected_blocks_concatenated.visual.vertex_colors = mustard

In [45]:
# create scene with the blocks on the target organ
target_with_tbs = trimesh.Scene([target, projected_blocks_concatenated])

# show
target_with_tbs.show()